# GTEx CellMarker validation

Mirrors `06_global_alignment.ipynb`, swapping the ground-truth gene-set resource from `GTEx_Tissues_V8_2023` (GTEx-bulk-derived, same cohort as the model's training samples) to **CellMarker 2024** (curated from independent single-cell/marker studies, unrelated to GTEx, and not used as a training prior for this CLAMP model — see `04_liver_disentangle` / Fig3 discussion for why GO:BP-based stats alone aren't independent validation).

Two analyses per GTEx tissue, each testing whether the selected SHAP LV(s) recover the true tissue by ORA against CellMarker 2024:

1. **top1**: the single highest-ranked SHAP LV per tissue.
2. **cumulative25**: the top-ranked LVs needed to reach `CUMULATIVE_PCT`% of cumulative SHAP per tissue, the same criterion used in `00_LV_importance_kmeans.ipynb` / `01_LV_importance_kmeans_biology.ipynb`.

In [1]:
library(here)
library(dplyr)
library(tidyr)
library(stringr)
library(clusterProfiler)


SHAP_DIR <- here('output', '03_model_biology', '01_gtex',
                 '02_rf_kmeans', '00_LV_importance_kmeans',
                 'gtex_feature_importance_kmeans_binary_shap')
OUT_DIR  <- here('output', '03_model_biology', '01_gtex', '02_rf_kmeans', '07_tissue_cellmarker_validation')
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

CLAMP_RDS       <- here('output', '01_model_building', '02_gtex', '01_CLAMP', 'CLAMPfull.rds')
CELLMARKER_FILE <- here('data', 'pathways', 'CellMarker_2024.txt')

N_LVS_PER_TISSUE <- 1L
CUMULATIVE_PCT   <- 25  # same threshold as 01_LV_importance_kmeans_biology.ipynb
TOP_GENE_PCT     <- 0.01
FDR_THRESH       <- 0.05

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses




Attaching package: ‘dplyr’




The following objects are masked from ‘package:stats’:

    filter, lag




The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




clusterProfiler v4.14.0 Learn more at https://yulab-smu.top/contribution-knowledge-mining/

Please cite:

G Yu. Thirteen years of clusterProfiler. The Innovation. 2024,
5(6):100722




Attaching package: ‘clusterProfiler’




The following object is masked from ‘package:stats’:

    filter




In [2]:
shap_all <- read.delim(file.path(SHAP_DIR, 'all_shap_positive.tsv'),
                       stringsAsFactors = FALSE, check.names = FALSE)

selected_lvs_top1 <- shap_all %>%
    dplyr::arrange(Tissue, Rank) %>%
    dplyr::group_by(Tissue) %>%
    dplyr::slice_head(n = N_LVS_PER_TISSUE) %>%
    dplyr::ungroup() %>%
    dplyr::rename(LV = Feature) %>%
    dplyr::select(Tissue, LV, Rank, Mean_SHAP_Tissue, Cumulative_SHAP, Cumulative_Percent)

selected_lvs_cum <- shap_all %>%
    dplyr::arrange(Tissue, Rank) %>%
    dplyr::group_by(Tissue) %>%
    dplyr::mutate(
        reaches_thresh = Cumulative_Percent >= CUMULATIVE_PCT,
        cutoff_rank = if (any(reaches_thresh)) min(Rank[reaches_thresh]) else max(Rank)
    ) %>%
    dplyr::filter(Rank <= cutoff_rank) %>%
    dplyr::ungroup() %>%
    dplyr::rename(LV = Feature) %>%
    dplyr::select(Tissue, LV, Rank, Mean_SHAP_Tissue, Cumulative_SHAP, Cumulative_Percent)

selections <- list(top1 = selected_lvs_top1, cumulative25 = selected_lvs_cum)

cat('Tissues:', dplyr::n_distinct(selected_lvs_top1$Tissue), '\n')
for (nm in names(selections)) {
    cat(sprintf('  [%s] selected LV/tissue rows: %d, unique LVs: %d\n',
                nm, nrow(selections[[nm]]), dplyr::n_distinct(selections[[nm]]$LV)))
}

dplyr::bind_rows(lapply(names(selections), function(nm) {
    selections[[nm]] %>%
        dplyr::count(Tissue, name = 'n_selected_lvs') %>%
        dplyr::mutate(analysis = nm)
})) %>%
    tidyr::pivot_wider(names_from = analysis, values_from = n_selected_lvs) %>%
    dplyr::arrange(Tissue)

Tissues: 23 


  [top1] selected LV/tissue rows: 23, unique LVs: 21
  [cumulative25] selected LV/tissue rows: 79, unique LVs: 69


Tissue,top1,cumulative25
<chr>,<int>,<int>
Adrenal Gland,1,2
Artery - Tibial,1,4
Cells - Cultured fibroblasts,1,4
Cells - EBV-transformed lymphocytes,1,6
Colon - Transverse,1,3
Esophagus - Mucosa,1,3
Heart - Atrial Appendage,1,2
Heart - Left Ventricle,1,6
Kidney - Cortex,1,2


In [3]:
clamp <- readRDS(CLAMP_RDS)
Z_full <- as.matrix(clamp$Z)
rm(clamp)

all_selected_lvs <- unique(unlist(lapply(selections, function(df) df$LV)))
selected_unique_lvs <- intersect(all_selected_lvs, colnames(Z_full))
missing_lvs <- setdiff(all_selected_lvs, colnames(Z_full))
if (length(missing_lvs) > 0) warning('Missing LVs in Z: ', paste(missing_lvs, collapse = ', '))

Z <- Z_full[, selected_unique_lvs, drop = FALSE]
rm(Z_full)
universe_genes <- rownames(Z)
n_top_genes <- max(1L, ceiling(TOP_GENE_PCT * nrow(Z)))

cat('Z:', nrow(Z), 'genes x', ncol(Z), 'unique LVs (union across analyses)\n')
cat('Top genes per LV:', n_top_genes, '\n')

Z: 21613 genes x 69 unique LVs (union across analyses)


Top genes per LV: 217 


## Load CellMarker 2024 and build the tissue keyword map

CellMarker 2024 (Enrichr GMT-style text, local copy) has no fixed delimiter between cell-type and tissue tokens in its set names (e.g. `Acinar Cell Pancreas Human`), unlike GTEx's clean suffix-stripping. Tissue assignment is done via a curated keyword/regex table instead, checked directly against the file beforehand. Two GTEx tissues — `Pituitary` and `Vagina` — have **zero** matching CellMarker Human sets; two more — `Cells - Cultured fibroblasts` and `Cells - EBV-transformed lymphocytes` — aren't real anatomical tissues and CellMarker has no "cultured"/"EBV-transformed" cell-line tag to match them. All four are kept in the tables below for direct comparison with `06_global_alignment.ipynb`, but will trivially show `tissue_correct = FALSE` (no ground-truth gene set exists to test against), which is flagged separately from a real non-replication.

In [4]:
read_gmt_human <- function(filename) {
    lines <- readLines(filename, warn = FALSE)
    gmt <- list()
    for (line in lines) {
        sp <- strsplit(line, '\t')[[1]]
        if (length(sp) < 3) next
        name  <- sp[1]
        if (!endsWith(name, ' Human')) next
        genes <- sp[3:length(sp)]
        genes <- genes[nzchar(genes)]
        if (length(genes) > 0) gmt[[name]] <- genes
    }
    gmt
}

cellmarker_t2g <- read_gmt_human(CELLMARKER_FILE)
cat('CellMarker 2024 (Human) sets:', length(cellmarker_t2g),
    '  unique genes:', length(unique(unlist(cellmarker_t2g))), '\n')

CellMarker 2024 (Human) sets: 1134   unique genes: 10164 


In [5]:
tissue_keywords <- c(
    'Adrenal Gland'                       = 'Adrenal',
    'Artery - Tibial'                     = '\\bArtery\\b',
    'Colon - Transverse'                  = '\\bColon\\b',
    'Esophagus - Mucosa'                  = 'Esophag',
    'Heart - Atrial Appendage'            = '\\bHeart\\b',
    'Heart - Left Ventricle'              = '\\bHeart\\b',
    'Kidney - Cortex'                     = '\\bKidney\\b',
    'Liver'                               = '\\bLiver\\b',
    'Lung'                                = '\\bLung\\b',
    'Muscle - Skeletal'                   = 'Skeletal Muscle',
    'Nerve - Tibial'                      = '\\bNerve\\b',
    'Ovary'                               = 'Ovar',
    'Pancreas'                            = 'Pancrea',
    'Prostate'                            = 'Prostate',
    'Small Intestine - Terminal Ileum'    = 'Intestine|Ileum',
    'Stomach'                             = '\\bStomach\\b',
    'Testis'                              = '\\bTesti(s|cular)',
    'Thyroid'                             = 'Thyroid',
    'Whole Blood'                         = '\\bBlood\\b'
)

normalize_text <- function(x) {
    x <- tolower(x)
    x <- gsub('[^a-z0-9]+', ' ', x)
    stringr::str_squish(x)
}

# A term can legitimately belong to MULTIPLE of our GTEx tissues when they share a
# keyword (e.g. Heart - Atrial Appendage / Heart - Left Ventricle both match 'Heart' --
# CellMarker doesn't distinguish heart sub-regions). Membership is many-to-many, not a
# single deterministic parsed_tissue per term.
term_names <- names(cellmarker_t2g)

tissue_matched_names <- lapply(tissue_keywords, function(pattern) {
    term_names[grepl(pattern, term_names, ignore.case = TRUE)]
})
names(tissue_matched_names) <- names(tissue_keywords)

term2gene_df <- do.call(rbind, lapply(term_names, function(term) {
    data.frame(term = term, gene = cellmarker_t2g[[term]], stringsAsFactors = FALSE)
}))

tissue_check <- dplyr::bind_rows(selections) %>%
    dplyr::distinct(Tissue) %>%
    dplyr::mutate(
        n_db_sets = vapply(Tissue, function(t) length(tissue_matched_names[[t]]), integer(1)),
        no_coverage = n_db_sets == 0
    ) %>%
    dplyr::arrange(n_db_sets)

cat('Tissues with NO CellMarker coverage (will always show tissue_correct = FALSE):\n')
print(tissue_check$Tissue[tissue_check$no_coverage])
tissue_check

Tissues with NO CellMarker coverage (will always show tissue_correct = FALSE):


[1] "Cells - Cultured fibroblasts"        "Cells - EBV-transformed lymphocytes"
[3] "Pituitary"                           "Vagina"                             


Tissue,n_db_sets,no_coverage
<chr>,<int>,<lgl>
Cells - Cultured fibroblasts,0,TRUE
Cells - EBV-transformed lymphocytes,0,TRUE
Pituitary,0,TRUE
Vagina,0,TRUE
Nerve - Tibial,1,FALSE
Muscle - Skeletal,3,FALSE
Thyroid,3,FALSE
Artery - Tibial,4,FALSE
Adrenal Gland,5,FALSE


## Run ORA per LV

`clusterProfiler::enricher` against the full ungrouped CellMarker 2024 term list (no pooling — exactly mirroring how `06_global_alignment.ipynb` keeps all individual GTEx sex/age-stratified terms separate), genes selected the same way: top `n_top_genes` by **decreasing raw Z value** per LV (matches `06_global_alignment.ipynb`'s convention, not absolute value).

In [6]:
run_cellmarker_ora <- function(genes, universe) {
    res <- tryCatch(
        clusterProfiler::enricher(
            gene          = genes,
            universe      = universe,
            TERM2GENE     = term2gene_df,
            pAdjustMethod = 'BH',
            pvalueCutoff  = 1,
            qvalueCutoff  = 1,
            minGSSize     = 10,
            maxGSSize     = 500
        ),
        error = function(e) NULL
    )
    if (is.null(res)) return(NULL)
    df <- as.data.frame(res)
    if (nrow(df) == 0) return(NULL)
    df
}

top_genes_per_lv <- lapply(colnames(Z), function(lv) {
    vals <- Z[, lv]
    universe_genes[order(vals, decreasing = TRUE)[seq_len(n_top_genes)]]
})
names(top_genes_per_lv) <- colnames(Z)

ora_list <- lapply(names(top_genes_per_lv), function(lv) {
    df <- run_cellmarker_ora(top_genes_per_lv[[lv]], universe_genes)
    if (is.null(df)) return(NULL)
    df$LV <- lv
    df
})

ora_all <- do.call(rbind, Filter(Negate(is.null), ora_list))
if (is.null(ora_all)) {
    ora_all <- data.frame()
} else {
    rownames(ora_all) <- NULL
    ora_all <- ora_all %>%
        dplyr::select(LV, ID, Description,
                      GeneRatio, BgRatio, pvalue, p.adjust, qvalue, geneID, Count)
}

write.csv(ora_all, file.path(OUT_DIR, 'cellmarker_ora_per_lv.csv'), row.names = FALSE)
cat('ORA rows:', nrow(ora_all), '\n')
head(ora_all)

ORA rows: 9982 


,LV,ID,Description,GeneRatio,BgRatio,pvalue,p.adjust,qvalue,geneID,Count
,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<int>
1,LV183,Leydig Cell Fetal Gonad Human,Leydig Cell Fetal Gonad Human,13/104,373/8989,0.0003485741,0.03497297,0.03221195,CYP17A1/GSTA3/RMDN2/MGARP/NR5A1/FGFR4/GSTA1/ALAS1/ACSF2/EMID1/GALM/PSAT1/EPHX1,13
2,LV183,Acinar Cell Pancreatic Islet Human,Acinar Cell Pancreatic Islet Human,6/104,91/8989,0.0006245173,0.03497297,0.03221195,GSTA1/KCTD14/REG3G/CPB1/PSAT1/C4B,6
3,LV183,Sertoli Cell Fetal Gonad Human,Sertoli Cell Fetal Gonad Human,12/104,420/8989,0.0032424162,0.12105021,0.11149361,AMHR2/FDXR/SERPINA5/CYP11A1/RDX/HS6ST1/GSTA1/KLHDC8B/PCED1B/REEP6/ZSWIM5/INHA,12
4,LV183,Sinusoidal Endothelial Cell Liver Human,Sinusoidal Endothelial Cell Liver Human,2/104,10/8989,0.0056156624,0.15258921,0.14054269,GPR182/FCN2,2
5,LV183,Lake Et al.Science.In3 Brain Human,Lake Et al.Science.In3 Brain Human,2/104,11/8989,0.0068120182,0.15258921,0.14054269,EMID1/SHISA8,2
6,LV183,Enterocyte Large Intestine Human,Enterocyte Large Intestine Human,3/104,43/8989,0.0132799109,0.20039624,0.18457549,KCNJ5/GSTA1/DGAT1,3


In [7]:
run_alignment_summary <- function(selected_lvs, ora_all, out_dir) {
    dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
    sig_ora <- ora_all %>% dplyr::filter(LV %in% selected_lvs$LV, p.adjust < FDR_THRESH)

    detail <- selected_lvs %>%
        dplyr::rowwise() %>%
        dplyr::mutate(
            own_terms = list(tissue_matched_names[[Tissue]]),
            n_sig_terms = sum(sig_ora$LV == LV),
            n_true_terms = sum(sig_ora$LV == LV & sig_ora$ID %in% unlist(own_terms)),
            tissue_correct = n_true_terms > 0,
            best_any_padj = {
                x <- sig_ora$p.adjust[sig_ora$LV == LV]
                if (length(x) == 0) NA_real_ else min(x, na.rm = TRUE)
            },
            best_true_padj = {
                x <- sig_ora$p.adjust[sig_ora$LV == LV & sig_ora$ID %in% unlist(own_terms)]
                if (length(x) == 0) NA_real_ else min(x, na.rm = TRUE)
            },
            matched_terms = paste(sig_ora$ID[sig_ora$LV == LV & sig_ora$ID %in% unlist(own_terms)], collapse = ' | ')
        ) %>%
        dplyr::ungroup() %>%
        dplyr::select(-own_terms)

    tissue_summary <- detail %>%
        dplyr::group_by(Tissue) %>%
        dplyr::summarise(
            n_selected_lvs = dplyr::n(),
            tissue_correct = any(tissue_correct),
            correct_lvs = paste(LV[tissue_correct], collapse = ';'),
            best_true_padj = if (all(is.na(best_true_padj))) NA_real_ else min(best_true_padj, na.rm = TRUE),
            .groups = 'drop'
        ) %>%
        dplyr::mutate(correct_score = as.integer(tissue_correct))

    final_pct_tissue_correct <- 100 * mean(tissue_summary$tissue_correct)
    final_summary <- data.frame(
        n_tissues = nrow(tissue_summary),
        n_tissues_correct = sum(tissue_summary$tissue_correct),
        pct_tissue_correct = final_pct_tissue_correct,
        stringsAsFactors = FALSE
    )

    write.csv(detail, file.path(out_dir, 'cellmarker_global_alignment_detail.csv'), row.names = FALSE)
    write.csv(tissue_summary, file.path(out_dir, 'cellmarker_global_alignment_summary.csv'), row.names = FALSE)
    write.csv(final_summary, file.path(out_dir, 'cellmarker_global_alignment_final_pct.csv'), row.names = FALSE)

    list(detail = detail, tissue_summary = tissue_summary, final_summary = final_summary)
}

results <- lapply(names(selections), function(nm) {
    run_alignment_summary(selections[[nm]], ora_all, file.path(OUT_DIR, nm))
})
names(results) <- names(selections)

dplyr::bind_rows(lapply(names(results), function(nm) {
    results[[nm]]$final_summary %>% dplyr::mutate(analysis = nm, .before = 1)
}))

analysis,n_tissues,n_tissues_correct,pct_tissue_correct
<chr>,<int>,<int>,<dbl>
top1,23,12,52.17391
cumulative25,23,14,60.86957


In [8]:
cat('--- top1 ---\n')
results$top1$tissue_summary %>% dplyr::arrange(Tissue)

cat('--- cumulative25 ---\n')
results$cumulative25$tissue_summary %>% dplyr::arrange(Tissue)

--- top1 ---


Tissue,n_selected_lvs,tissue_correct,correct_lvs,best_true_padj,correct_score
<chr>,<int>,<lgl>,<chr>,<dbl>,<int>
Adrenal Gland,1,FALSE,,NA,0
Artery - Tibial,1,FALSE,,NA,0
Cells - Cultured fibroblasts,1,FALSE,,NA,0
Cells - EBV-transformed lymphocytes,1,FALSE,,NA,0
Colon - Transverse,1,TRUE,LV562,3.318123e-05,1
Esophagus - Mucosa,1,TRUE,LV84,1.764409e-04,1
Heart - Atrial Appendage,1,TRUE,LV420,1.661363e-17,1
Heart - Left Ventricle,1,TRUE,LV505,2.095422e-13,1
Kidney - Cortex,1,TRUE,LV229,1.068030e-03,1


--- cumulative25 ---


Tissue,n_selected_lvs,tissue_correct,correct_lvs,best_true_padj,correct_score
<chr>,<int>,<lgl>,<chr>,<dbl>,<int>
Adrenal Gland,2,FALSE,,NA,0
Artery - Tibial,4,TRUE,LV563;LV90;LV387;LV37,5.607692e-03,1
Cells - Cultured fibroblasts,4,FALSE,,NA,0
Cells - EBV-transformed lymphocytes,6,FALSE,,NA,0
Colon - Transverse,3,TRUE,LV562;LV368;LV108,4.745155e-10,1
Esophagus - Mucosa,3,TRUE,LV84;LV330;LV81,1.764409e-04,1
Heart - Atrial Appendage,2,TRUE,LV420;LV509,1.661363e-17,1
Heart - Left Ventricle,6,TRUE,LV505;LV554;LV76;LV137;LV9;LV203,2.095422e-13,1
Kidney - Cortex,2,TRUE,LV229;LV50,1.768063e-08,1


In [9]:
detail_cols <- c('Tissue', 'LV', 'Rank', 'Mean_SHAP_Tissue', 'Cumulative_Percent',
                 'tissue_correct', 'n_sig_terms', 'n_true_terms',
                 'best_true_padj', 'matched_terms')

cat('--- top1 ---\n')
results$top1$detail %>% dplyr::select(dplyr::all_of(detail_cols)) %>% dplyr::arrange(Tissue, Rank)

cat('--- cumulative25 ---\n')
results$cumulative25$detail %>% dplyr::select(dplyr::all_of(detail_cols)) %>% dplyr::arrange(Tissue, Rank)

--- top1 ---


Tissue,LV,Rank,Mean_SHAP_Tissue,Cumulative_Percent,tissue_correct,n_sig_terms,n_true_terms,best_true_padj,matched_terms
<chr>,<chr>,<int>,<dbl>,<dbl>,<lgl>,<int>,<int>,<dbl>,<chr>
Adrenal Gland,LV183,1,0.13753360,14.365823,FALSE,2,0,NA,
Artery - Tibial,LV563,1,0.08563937,9.142689,FALSE,0,0,NA,
Cells - Cultured fibroblasts,LV546,1,0.11603051,12.029419,FALSE,11,0,NA,
Cells - EBV-transformed lymphocytes,LV21,1,0.06096849,6.129557,FALSE,8,0,NA,
Colon - Transverse,LV562,1,0.10610394,11.353268,TRUE,9,2,3.318123e-05,Epithelial Cell Colon Human | Progenitor Cell Colon Human
Esophagus - Mucosa,LV84,1,0.13713745,14.194683,TRUE,7,1,1.764409e-04,Secretory Progenitor Cell Esophagus Human
Heart - Atrial Appendage,LV420,1,0.16581111,17.256570,TRUE,1,1,1.661363e-17,Cardiomyocyte Heart Human
Heart - Left Ventricle,LV505,1,0.07538633,7.857362,TRUE,2,1,2.095422e-13,Cardiomyocyte Heart Human
Kidney - Cortex,LV229,1,0.15278041,16.221166,TRUE,7,2,1.068030e-03,Nephron Epithelial Cell Kidney Human | Proximal Tubular Cell Kidney Human


--- cumulative25 ---


Tissue,LV,Rank,Mean_SHAP_Tissue,Cumulative_Percent,tissue_correct,n_sig_terms,n_true_terms,best_true_padj,matched_terms
<chr>,<chr>,<int>,<dbl>,<dbl>,<lgl>,<int>,<int>,<dbl>,<chr>
Adrenal Gland,LV183,1,0.13753360,14.365823,FALSE,2,0,NA,
Adrenal Gland,LV3,2,0.13109711,28.059334,FALSE,4,0,NA,
Artery - Tibial,LV563,1,0.08563937,9.142689,FALSE,0,0,NA,
Artery - Tibial,LV90,2,0.07622058,17.279846,FALSE,0,0,NA,
Artery - Tibial,LV387,3,0.05017576,22.636511,TRUE,4,1,5.607692e-03,Endothelial Cell Artery Human
Artery - Tibial,LV37,4,0.04984620,27.957992,FALSE,0,0,NA,
Cells - Cultured fibroblasts,LV546,1,0.11603051,12.029419,FALSE,11,0,NA,
Cells - Cultured fibroblasts,LV91,2,0.07353234,19.652857,FALSE,128,0,NA,
Cells - Cultured fibroblasts,LV363,3,0.03988249,23.787658,FALSE,0,0,NA,


## Per-LV correctness within the cumulative25 selection

Not just "any LV in the tissue's selection is correct" but the % of individual LVs that, on their own, recover the true tissue — the stricter, more informative number for the manuscript.

In [10]:
per_lv_summary <- results$cumulative25$detail %>%
    dplyr::group_by(Tissue) %>%
    dplyr::summarise(
        n_lvs = dplyr::n(),
        n_correct = sum(tissue_correct),
        pct_correct = 100 * n_correct / n_lvs,
        .groups = 'drop'
    ) %>%
    dplyr::arrange(pct_correct)

print(per_lv_summary, n = Inf)

total_lvs <- sum(per_lv_summary$n_lvs)
total_correct <- sum(per_lv_summary$n_correct)
cat(sprintf(
    '\nOverall: %d/%d LV-tissue rows correct (%.1f%%) across cumulative25 selection\n',
    total_correct, total_lvs, 100 * total_correct / total_lvs
))
cat(sprintf('Tissues at 100%% LV concordance: %d/%d\n',
            sum(per_lv_summary$pct_correct == 100), nrow(per_lv_summary)))

write.csv(per_lv_summary, file.path(OUT_DIR, 'cumulative25', 'cellmarker_global_alignment_per_lv_pct.csv'), row.names = FALSE)

# A tibble: 23 × 4
   Tissue                              n_lvs n_correct pct_correct
   <chr>                               <int>     <int>       <dbl>
 1 Adrenal Gland                           2         0         0  
 2 Cells - Cultured fibroblasts            4         0         0  
 3 Cells - EBV-transformed lymphocytes     6         0         0  
 4 Muscle - Skeletal                       3         0         0  
 5 Nerve - Tibial                          3         0         0  
 6 Ovary                                   3         0         0  
 7 Pituitary                               5         0         0  
 8 Prostate                                4         0         0  
 9 Vagina                                  3         0         0  
10 Artery - Tibial                         4         1        25  
11 Testis                                  4         1        25  
12 Whole Blood                             4         1        25  
13 Esophagus - Mucosa                      


Overall: 30/79 LV-tissue rows correct (38.0%) across cumulative25 selection


Tissues at 100% LV concordance: 7/23


## Liver example detail

In [11]:
results$cumulative25$detail %>%
    dplyr::filter(Tissue == 'Liver') %>%
    dplyr::select(dplyr::all_of(detail_cols)) %>%
    dplyr::arrange(Rank)

Tissue,LV,Rank,Mean_SHAP_Tissue,Cumulative_Percent,tissue_correct,n_sig_terms,n_true_terms,best_true_padj,matched_terms
<chr>,<chr>,<int>,<dbl>,<dbl>,<lgl>,<int>,<int>,<dbl>,<chr>
Liver,LV21,1,0.11317176,11.56292,TRUE,8,2,7.594161e-09,Liver Bud Hepatic Cell Liver Human | Hepatocyte Liver Human
Liver,LV455,2,0.08424926,20.17079,TRUE,4,2,5.145906e-10,Liver Bud Hepatic Cell Liver Human | Hepatocyte Liver Human
Liver,LV59,3,0.07419744,27.75165,TRUE,7,2,9.198730e-09,Hepatocyte Liver Human | Liver Bud Hepatic Cell Liver Human


In [12]:
stopifnot(file.exists(file.path(OUT_DIR, 'cellmarker_ora_per_lv.csv')))

for (nm in names(selections)) {
    out_dir <- file.path(OUT_DIR, nm)
    expected_rows <- nrow(selections[[nm]])
    stopifnot(nrow(results[[nm]]$detail) == expected_rows)
    stopifnot(nrow(results[[nm]]$tissue_summary) == dplyr::n_distinct(shap_all$Tissue))
    stopifnot(file.exists(file.path(out_dir, 'cellmarker_global_alignment_detail.csv')))
    stopifnot(file.exists(file.path(out_dir, 'cellmarker_global_alignment_summary.csv')))
    stopifnot(file.exists(file.path(out_dir, 'cellmarker_global_alignment_final_pct.csv')))
    cat(sprintf(
        '[%s] Checks passed. Denominator = %d tissues and %d selected LV/tissue rows. Final %% tissue correct = %.2f\n',
        nm, nrow(results[[nm]]$tissue_summary), nrow(results[[nm]]$detail),
        results[[nm]]$final_summary$pct_tissue_correct
    ))
}

[top1] Checks passed. Denominator = 23 tissues and 23 selected LV/tissue rows. Final % tissue correct = 52.17
[cumulative25] Checks passed. Denominator = 23 tissues and 79 selected LV/tissue rows. Final % tissue correct = 60.87
